# 04 · Evaluation
Test-set metrics for the trained model (`python -m src.evaluate`) and Keras-vs-TFLite agreement.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # run from ml/notebooks
import numpy as np
import matplotlib.pyplot as plt
from src import config
import json
from IPython.display import Image, display

In [ ]:
metrics = json.loads(config.METRICS_PATH.read_text())
print('accuracy', metrics['test_accuracy'], 'macro-F1', metrics['macro_f1'])
for k, v in metrics['per_class'].items():
    print(f"{k:20s} P={v['precision']:.3f} R={v['recall']:.3f} F1={v['f1-score']:.3f}")

In [ ]:
display(Image(str(config.REPORTS_DIR / 'confusion_matrix.png')))
display(Image(str(config.REPORTS_DIR / 'training_curves.png')))

In [ ]:
import tensorflow as tf
from src.data_loader import load_split
X_test, y_test, _ = load_split('test')
interp = tf.lite.Interpreter(model_path=str(config.TFLITE_MODEL_PATH)); interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
preds = []
for x in X_test[:500]:
    interp.set_tensor(inp['index'], x[None].astype('float32')); interp.invoke(); preds.append(interp.get_tensor(out['index'])[0].argmax())
print('TFLite accuracy on first 500 test windows:', (np.array(preds) == y_test[:500]).mean())